# CAWOT-CM coreset sweep (V0 + V1 + V2) — full-rigor protocol

**Plan-faithful full sweep:**
- 5 methods: `random` / `v0` / `v0_proto` / `v1` / **`v2`** (the full plan method)
- 6 budgets {5, 10, 20, 30, 40, 50}% (Part B2 scaling curve)
- 3 seeds (42, 1, 2) for error bars (Tuần 4-5)
- Per-category eval: **goal / full / wentwrong** (wentwrong = anomaly)
- Resumable (records.csv persisted)

**Total: 5 × 6 × 3 = 90 fine-tune runs ≈ 20-21 h.** Split across ~3-4 Kaggle sessions.

**V2 adds**: Q_proxy text queries (re-encoded with CLIP-B/16) → Wasserstein-aware budget allocation per cluster.

Setup: GPU **P100** + Internet **ON**.

## 1. Env + clone + install

In [ ]:
!nvidia-smi -L
import os
if not os.path.exists("/kaggle/working/cawot-cm"):
    !git clone https://github.com/HohoHocCode/cawot-cm.git /kaggle/working/cawot-cm
%cd /kaggle/working/cawot-cm
!pip install -q open_clip_torch faiss-gpu-cu12 einops huggingface_hub gdown

## 2. Download train shards from HuggingFace (~7 GB / 14 GB extracted)

In [ ]:
!python scripts/setup_data.py --output /kaggle/working/pab_data --num-shards 5

## 3. Download Q_proxy from Google Drive (V2 only — 2,593 LLM queries, <1 MB)

Friend's qproxy folder on Drive contains EVA02-1024d embeddings we cannot reuse directly (pool is CLIP-512d). We download only `queries.json` and re-encode captions with CLIP-B/16 text encoder so everything is in one embedding space.

In [ ]:
!python scripts/setup_qproxy.py --output /kaggle/working/qproxy --only-queries

## 4. Sanity check that one image resolves + Q_proxy loads

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/cawot-cm")
from src.data import build_pool, TrainPoolDataset
from src.qproxy import load_qproxy_captions
from torchvision import transforms
anns, shard_roots = build_pool("/kaggle/working/pab_data/annotations",
                               "/kaggle/working/pab_data/images", sample_size=100, seed=42)
print(f"pool: {len(anns)} samples; shards: {sorted(shard_roots)}")
tx = transforms.Compose([transforms.Resize(224), transforms.CenterCrop(224), transforms.ToTensor()])
s = TrainPoolDataset(anns, shard_roots, image_transform=tx)[0]
print(f"image {s['image'].shape}; caption: {s['caption'][:80]}...")
qcaps = load_qproxy_captions("/kaggle/working/qproxy/queries.json")
print(f"Q_proxy: {len(qcaps)} queries; first: {qcaps[0][:100]}...")
print("\u2713 everything resolves")

## 5. Run the full sweep (resumable)

On Kaggle P100:
- Embeddings (image + text, 50K): ~18-22 min (cached after)
- Q_proxy encoding (2,593 captions): ~10 sec (cached after)
- 90 fine-tune runs (5 methods × 6 budgets × 3 seeds): ~18-19 h
- **Total ~20-21 h across 3-4 Kaggle sessions**

**Multi-session strategy:** keep `train.seeds: [42, 1, 2]` in config. Restart this cell after each session timeout — already-completed combos are skipped via `records.csv`.

In [ ]:
!python scripts/run_sweep.py --config config.yaml

## 6. Results table (overall + per-category)

In [ ]:
import json, pandas as pd
summary = json.load(open("/kaggle/working/outputs/eval/summary.json"))
print("zeroshot:", summary["zeroshot"])

methods = [m for m in ["random", "v0", "v0_proto", "v1", "v2"] if m in summary]
budgets = sorted({float(b) for m in methods for b in summary[m].keys()})
categories = ["overall", "goal", "full", "wentwrong"]

rows = []
for c in categories:
    for b in budgets:
        r = {"category": c, "budget": f"{int(b*100)}%"}
        for m in methods:
            s = summary[m].get(str(b), {}).get(c)
            r[m] = f"{s['mean_R@1_mean']:.2f}±{s['mean_R@1_std']:.2f}" if s else "-"
        rows.append(r)
df = pd.DataFrame(rows)
df

## 7. Plot R@1 vs budget — overall + wentwrong (anomaly subset)

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
markers = {"random": "o", "v0": "^", "v0_proto": "v", "v1": "s", "v2": "D"}
for ax, cat in zip(axes, ["overall", "wentwrong"]):
    x = [b * 100 for b in budgets]
    for m in methods:
        y = [summary[m].get(str(b), {}).get(cat, {}).get("mean_R@1_mean") for b in budgets]
        e = [summary[m].get(str(b), {}).get(cat, {}).get("mean_R@1_std", 0) for b in budgets]
        if all(v is None for v in y): continue
        ax.errorbar(x, y, yerr=e, marker=markers.get(m, "o"), capsize=3, label=m)
    zs = summary["zeroshot"].get(cat)
    if zs is not None:
        ax.axhline(zs, ls="--", c="gray", label=f"zero-shot ({zs:.1f})")
    ax.set_xlabel("Budget (% of train pool)"); ax.set_ylabel("mean R@1")
    ax.set_title(f"{cat}"); ax.legend(); ax.grid(alpha=0.3)
fig.suptitle("V0 family + V1 + V2 — Overall vs. Anomaly (wentwrong)")
plt.tight_layout()
plt.savefig("/kaggle/working/outputs/eval/sweep_curve.png", dpi=120, bbox_inches="tight")
plt.show()

## 8. How to read for the report

**V2 expected behavior**: Wasserstein-aware budget pushes more samples toward clusters that don't look like Q_proxy queries → expected to lift anomaly retrieval (wentwrong) especially at low budget, since anomaly clusters likely have higher W2 to Q_proxy.

Three plausible outcomes for V2:
- **V2 > V1 ≥ v0_proto on wentwrong** → the headline contribution: budget allocation matters more than within-cluster selection for anomaly. Strongest paper story.
- **V2 ≈ V1 across the board** → Q_proxy alignment with anomaly clusters wasn't strong enough; ablation shows budget allocation is neutral. Still valid V2 finding.
- **V2 < V1** → Wasserstein-driven budget hurts (over-allocates to noisy clusters). Re-tune `v2_alpha` or filter Q_proxy.

Artifacts to commit:
- `outputs/eval/records.csv` (360+ rows: 5 methods × 6 budgets × 3 seeds × 4 categories)
- `outputs/eval/summary.json`
- `outputs/eval/sweep_curve.png`